# Model Validation & Training
Sections 1–7: sanity-check forward/backward passes, gradients, optimizer, and memory.
Sections 8–10: run the full training experiments (fALFF, GM, multi-modal) overnight.

In [ ]:
# Standard imports and device selection.
# MPS = Apple Silicon GPU; falls back to CPU if unavailable.
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn

from models.single_modal_3d import fmri_cnn, smri_cnn
from models.multi_modal_3d import MultiModal3DCNN
from utils.dataset import build_datasets
from torch.utils.data import DataLoader

DATA_DIR = '../data/raw'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Forward Pass on Real Data
Run an actual subject volume (not dummy zeros) through each model.

In [ ]:
# Load the three single-modal datasets and one multi-modal dataset.
# BatchNorm1d requires >1 sample during training, so we always use batch=2 here.
ds_falff = build_datasets(DATA_DIR, derivative='falff')
ds_gm    = build_datasets(DATA_DIR, derivative='gm')
ds_multi = build_datasets(DATA_DIR, multi_modal=True)

# Stack two real subject volumes into a batch.
fmri_vol = torch.stack([ds_falff[0][0], ds_falff[1][0]])  # (2, 1, 47, 60, 46)
gm_vol   = torch.stack([ds_gm[0][0],   ds_gm[1][0]])      # (2, 1, 90, 117, 100)
labels   = torch.tensor([ds_falff[0][1], ds_falff[1][1]])

model_f = fmri_cnn().eval()
model_s = smri_cnn().eval()

with torch.no_grad():
    out_f = model_f(fmri_vol)
    out_s = model_s(gm_vol)

print(f'fMRI logits: {out_f}')
print(f'sMRI logits: {out_s}')
print(f'fMRI predicted: {out_f.argmax(dim=1).tolist()}  labels: {labels.tolist()}  (0=ADHD, 1=TDC)')

## 2. Loss Computation
CrossEntropyLoss on real data — should produce a finite scalar near log(2) ≈ 0.693 for an untrained model.

In [ ]:
# Verify that the loss is finite and close to the random-init baseline (log 2 ≈ 0.693).
# A NaN or Inf here would indicate a numerical issue in the model (e.g. div-by-zero in BN).
loss_fn = nn.CrossEntropyLoss()

model_f.train()
out  = model_f(fmri_vol)   # batch of 2 — BN works
loss = loss_fn(out, labels)

print(f'Loss: {loss.item():.4f}  (expect ~0.693 for random init)')
print(f'Finite: {loss.isfinite().item()}')

## 3. Backward Pass — Gradient Flow
Check that gradients reach every layer, including early conv weights.

In [ ]:
# If any layer shows NO GRADIENT, the backward graph is broken for that branch.
# All parameters (conv weights, BN gamma/beta, FC weights/biases) should have non-zero norms.
model_f.zero_grad()
out  = model_f(fmri_vol)
loss = loss_fn(out, labels)
loss.backward()

print('Gradient norms per layer:')
for name, p in model_f.named_parameters():
    if p.grad is not None:
        print(f'  {name:50s}  grad_norm={p.grad.norm():.6f}')
    else:
        print(f'  {name:50s}  NO GRADIENT')

## 4. SGD Optimizer Step
Confirm the paper's exact optimizer settings work without errors.

In [ ]:
# Paper: SGD lr=1e-4, momentum=0.9, StepLR step_size=20, gamma=0.5.
# After 20 scheduler steps the LR should halve from 1e-4 to 5e-5.
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR

model_f.train()
optimizer = SGD(model_f.parameters(), lr=0.0001, momentum=0.9)
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)

for step in range(3):
    optimizer.zero_grad()
    out  = model_f(fmri_vol)
    loss = loss_fn(out, labels)
    loss.backward()
    optimizer.step()
    print(f'Step {step+1}: loss={loss.item():.4f}  lr={scheduler.get_last_lr()}')

for _ in range(20): scheduler.step()
print(f'LR after 20 scheduler steps: {scheduler.get_last_lr()}  (expect 0.00005)')

## 5. Train vs Eval Mode
Dropout should be active in train mode and off in eval mode — outputs should differ.

In [ ]:
# In train mode, Dropout(0.5) randomly zeroes half the neurons, so two identical forward
# passes produce different logits.  In eval mode Dropout is disabled and outputs are stable.
torch.manual_seed(0)
model_f.train()
with torch.no_grad():
    out_train_1 = model_f(fmri_vol)
    out_train_2 = model_f(fmri_vol)

model_f.eval()
with torch.no_grad():
    out_eval_1 = model_f(fmri_vol)
    out_eval_2 = model_f(fmri_vol)

print('Train mode — two passes should differ (dropout active):')
print(f'  same={torch.allclose(out_train_1, out_train_2)}')

print('Eval mode — two passes should be identical (dropout off):')
print(f'  same={torch.allclose(out_eval_1, out_eval_2)}')

## 6. Batch Size Memory Check
The paper uses batch_size=20. Test the largest batch that fits in memory.

In [ ]:
# With only 79 subjects and 4-fold CV, batch_size=8 avoids wasting subjects via drop_last.
# This cell confirms all batch sizes up to 20 fit in memory (or shows the OOM boundary).
import traceback

model_f.train()
optimizer.zero_grad()

for bs in [4, 8, 16, 20]:
    try:
        batch = torch.zeros(bs, 1, 47, 60, 46)
        labels = torch.zeros(bs, dtype=torch.long)
        out  = model_f(batch)
        loss = loss_fn(out, labels)
        loss.backward()
        optimizer.zero_grad()
        print(f'  batch_size={bs:2d}  OK  loss={loss.item():.4f}')
    except RuntimeError as e:
        print(f'  batch_size={bs:2d}  OOM: {e}')

## 7. Multi-modal Backward Pass

In [ ]:
# Confirm that the fused fMRI+sMRI model trains end-to-end without errors.
# Both branches share the same loss signal — gradients must flow through the concat layer.
multi_model = MultiModal3DCNN()
multi_model.train()
optimizer_m = SGD(multi_model.parameters(), lr=0.0001, momentum=0.9)

fmri_b = torch.stack([ds_multi[0][0], ds_multi[1][0]])
smri_b = torch.stack([ds_multi[0][1], ds_multi[1][1]])
labels_b = torch.tensor([ds_multi[0][2], ds_multi[1][2]])

optimizer_m.zero_grad()
out  = multi_model(fmri_b, smri_b)
loss = loss_fn(out, labels_b)
loss.backward()
optimizer_m.step()

print(f'Multi-modal loss: {loss.item():.4f}  finite={loss.isfinite().item()}')
print(f'Output: {out.tolist()}')

---
## 8. Training Configuration
Set `N_REPEATS=2` and `EPOCHS=5` for a quick smoke-test; use `50` and `100` for the full overnight run.

In [ ]:
# All three experiments below use these shared hyperparameters.
# Full paper settings: N_REPEATS=50, EPOCHS=100.
# Smoke-test settings: N_REPEATS=2, EPOCHS=5.
import subprocess, os, json, re, ast
from pathlib import Path

SCRIPTS_DIR = Path('../scripts')
OUTPUTS_DIR = Path('../outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

N_REPEATS  = 50    # change to 2 for smoke-test
N_FOLDS    = 4
EPOCHS     = 100   # change to 5 for smoke-test
BATCH_SIZE = 8     # reduced from paper's 20 — avoids wasting subjects via drop_last

print(f'N_REPEATS={N_REPEATS}  EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}')
print(f'Estimated time at ~21s / (2 repeats × 5 epochs): ~{N_REPEATS * EPOCHS * 21 / (2*5) / 3600:.1f} hours per experiment')

In [ ]:
# Helper that launches train.py as a subprocess, streams stdout live,
# and parses the final summary lines into a dict.
def run_experiment(label, extra_args):
    out_dir = OUTPUTS_DIR / label
    cmd = [
        sys.executable, str(SCRIPTS_DIR / 'train.py'),
        '--n-repeats',  str(N_REPEATS),
        '--n-folds',    str(N_FOLDS),
        '--epochs',     str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--output-dir', str(out_dir),
    ] + extra_args

    print(f'\n{"="*60}')
    print(f'EXPERIMENT: {label}')
    print(f'CMD: {" ".join(cmd)}')
    print('='*60)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(SCRIPTS_DIR),
    )
    lines = []
    for line in proc.stdout:
        print(line, end='', flush=True)  # stream output in real-time
        lines.append(line)
    proc.wait()

    # Extract the three summary numbers from the final printed lines.
    summary = {'label': label, 'returncode': proc.returncode}
    for line in lines:
        m = re.search(r'Mean val accuracy\s*:\s*([\d.]+)%', line)
        if m: summary['mean_val_acc'] = float(m.group(1))
        m = re.search(r'Variance\s*:\s*([\d.]+)', line)
        if m: summary['variance'] = float(m.group(1))
        m = re.search(r'Best single run\s*:\s*([\d.]+)%', line)
        if m: summary['best_run'] = float(m.group(1))
    return summary

all_results = []  # collect summaries across all three experiments
print('Helper ready.')

## 9. Run Experiments
Cells below run sequentially. Each one takes O(hours) for the full 50×100 setting.

In [ ]:
# Experiment A: single-modal fALFF.
# Paper reports 62.06% mean val accuracy (Table III).
r = run_experiment('falff', ['--mode', 'single', '--derivative', 'falff'])
all_results.append(r)
print(f"\n→ fALFF result: {r.get('mean_val_acc', 'N/A')}%  (paper: 62.06%)")

In [ ]:
# Experiment B: single-modal GM (structural MRI grey-matter density).
# Paper reports 65.43% mean val accuracy (Table III).
r = run_experiment('gm', ['--mode', 'single', '--derivative', 'gm'])
all_results.append(r)
print(f"\n→ GM result: {r.get('mean_val_acc', 'N/A')}%  (paper: 65.43%)")

In [ ]:
# Experiment C: multi-modal fALFF + GM — paper's best configuration.
# Fuses 512-dim feature vectors from both branches, feeds 1024-dim vector into a shared FC head.
# Paper reports 69.15% mean val accuracy (Table III).
r = run_experiment('falff_gm_multi', [
    '--mode', 'multi',
    '--fmri-derivative', 'falff',
    '--smri-derivative', 'gm',
])
all_results.append(r)
print(f"\n→ Multi-modal result: {r.get('mean_val_acc', 'N/A')}%  (paper: 69.15%)")

## 10. Summary vs. Paper

In [ ]:
# Compare our results to the paper's Table III.
# Δ < 0 means we're below the paper; Δ > 0 means we exceeded it.
import pandas as pd
import matplotlib.pyplot as plt

paper_targets = {
    'falff':         62.06,
    'gm':            65.43,
    'falff_gm_multi':69.15,
}

rows = []
for r in all_results:
    ours  = r.get('mean_val_acc', float('nan'))
    paper = paper_targets.get(r['label'], float('nan'))
    rows.append({
        'Experiment':   r['label'],
        'Our acc (%)':  f"{ours:.2f}",
        'Paper (%)':    f"{paper:.2f}",
        'Δ':            f"{ours - paper:+.2f}",
        'Variance':     f"{r.get('variance', float('nan')):.2f}",
        'Best run (%)': f"{r.get('best_run', float('nan')):.2f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save full summary for the report
summary_path = OUTPUTS_DIR / 'summary.json'
with open(summary_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved to {summary_path}')

In [ ]:
# Plot per-repeat mean val accuracy for each experiment.
# Each point is the mean of 4 folds for that repeat.
# Dashed lines show the paper's reported targets.
fig, ax = plt.subplots(figsize=(11, 4))

colors = {'falff': 'C0', 'gm': 'C1', 'falff_gm_multi': 'C2'}

for r in all_results:
    results_file = OUTPUTS_DIR / r['label'] / 'results.txt'
    if not results_file.exists():
        continue
    with open(results_file) as f:
        for line in f:
            if line.startswith('all_repeat_means:'):
                vals = ast.literal_eval(line.split(':', 1)[1].strip())
                c = colors.get(r['label'], None)
                ax.plot(range(1, len(vals)+1), vals, marker='.', color=c, label=r['label'])
                # paper baseline as dashed line in the same colour
                ax.axhline(
                    paper_targets.get(r['label'], 0),
                    linestyle='--', color=c, alpha=0.5,
                    label=f"{r['label']} paper target",
                )

ax.set_xlabel('Repeat')
ax.set_ylabel('Mean val accuracy (%)')
ax.set_title('Validation accuracy per repeat — all experiments')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()